In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Load clean data
train = pd.read_csv('../data/processed/train_clean.csv')

print("Shape:", train.shape)
print("Columns:", list(train.columns))
train.head()

Shape: (20631, 17)
Columns: ['engine_id', 'cycle', 'RUL', 'sensor2', 'sensor3', 'sensor4', 'sensor7', 'sensor8', 'sensor9', 'sensor11', 'sensor12', 'sensor13', 'sensor14', 'sensor15', 'sensor17', 'sensor20', 'sensor21']


,engine_id,cycle,RUL,sensor2,sensor3,sensor4,sensor7,sensor8,sensor9,sensor11,sensor12,sensor13,sensor14,sensor15,sensor17,sensor20,sensor21
0,1,1,191,641.82,1589.70,1400.60,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392,39.06,23.4190
1,1,2,190,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236
2,1,3,189,642.35,1587.99,1404.20,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442
3,1,4,188,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739
4,1,5,187,642.37,1582.85,1406.22,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044


In [2]:
# Define sensor columns
sensor_cols = ['sensor2', 'sensor3', 'sensor4', 'sensor7', 'sensor8', 
               'sensor9', 'sensor11', 'sensor12', 'sensor13', 'sensor14', 
               'sensor15', 'sensor17', 'sensor20', 'sensor21']

# Sort by engine and cycle — critical for time-series features
train = train.sort_values(['engine_id', 'cycle']).reset_index(drop=True)

# Rolling window size
window = 5

# Build features per engine
feature_dfs = []

for engine_id, group in train.groupby('engine_id'):
    group = group.copy()
    
    for sensor in sensor_cols:
        # Rolling mean — smooths out noise
        group[f'{sensor}_roll_mean'] = group[sensor].rolling(window, min_periods=1).mean()
        
        # Rolling std — captures variability
        group[f'{sensor}_roll_std'] = group[sensor].rolling(window, min_periods=1).std().fillna(0)
        
        # Lag feature — what was the value 1 cycle ago
        group[f'{sensor}_lag1'] = group[sensor].shift(1).fillna(group[sensor].iloc[0])
        
        # Rate of change — how fast is the sensor changing
        group[f'{sensor}_roc'] = group[sensor].diff(1).fillna(0)
    
    feature_dfs.append(group)

# Combine all engines
train_featured = pd.concat(feature_dfs, ignore_index=True)

print("Shape after feature engineering:", train_featured.shape)
print("Total features created:", train_featured.shape[1] - 3)  # minus engine_id, cycle, RUL
train_featured.head()

Shape after feature engineering: (20631, 73)
Total features created: 70


,engine_id,cycle,RUL,sensor2,sensor3,sensor4,sensor7,sensor8,sensor9,sensor11,...,sensor17_lag1,sensor17_roc,sensor20_roll_mean,sensor20_roll_std,sensor20_lag1,sensor20_roc,sensor21_roll_mean,sensor21_roll_std,sensor21_lag1,sensor21_roc
0,1,1,191,641.82,1589.70,1400.60,554.36,2388.06,9046.19,47.47,...,392.0,0.0,39.060000,0.000000,39.06,0.00,23.419000,0.000000,23.4190,0.0000
1,1,2,190,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,...,392.0,0.0,39.030000,0.042426,39.06,-0.06,23.421300,0.003253,23.4190,0.0046
2,1,3,189,642.35,1587.99,1404.20,554.26,2388.08,9052.94,47.27,...,392.0,-2.0,39.003333,0.055076,39.00,-0.05,23.395600,0.044573,23.4236,-0.0794
3,1,4,188,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,...,390.0,2.0,38.972500,0.076322,38.95,-0.07,23.390175,0.037977,23.3442,0.0297
4,1,5,187,642.37,1582.85,1406.22,554.00,2388.06,9055.15,47.28,...,392.0,1.0,38.958000,0.073621,38.88,0.02,23.393020,0.033498,23.3739,0.0305


In [3]:
# Separate features and target
feature_cols = [col for col in train_featured.columns 
                if col not in ['engine_id', 'cycle', 'RUL']]

X = train_featured[feature_cols].copy()
y = train_featured['RUL'].copy()

# Normalize features using MinMaxScaler
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

print("Feature matrix shape:", X_scaled.shape)
print("Target shape:", y.shape)
print("\nSample scaled values (first 3 rows, first 5 cols):")
print(X_scaled.iloc[:3, :5].round(4))

# Save everything
train_featured.to_csv('../data/processed/train_featured.csv', index=False)
X_scaled.to_csv('../data/processed/X_scaled.csv', index=False)
y.to_csv('../data/processed/y.csv', index=False)

# Save scaler for later use in deployment
import pickle
with open('../data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("\nAll files saved to data/processed/")

Feature matrix shape: (20631, 70)
Target shape: (20631,)

Sample scaled values (first 3 rows, first 5 cols):
   sensor2  sensor3  sensor4  sensor7  sensor8
0   0.1837   0.4068   0.3098   0.7262   0.2424
1   0.2831   0.4530   0.3526   0.6280   0.2121
2   0.3434   0.3695   0.3705   0.7101   0.2727

All files saved to data/processed/
